In [ ]:
!pip install apache_beam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.0/152.0 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.5/261.5 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from apache_beam.pvalue import AsList
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

# Initialize Spark session
spark = SparkSession.builder.appName("BeamToPySpark").getOrCreate()

# Function to store data from Beam pipeline
class CaptureResults(beam.DoFn):
    def __init__(self, name):
        self.name = name

    def process(self, element):
        global data_captured
        data_captured[self.name].append(element)

data_captured = {"total_salary": [], "avg_salary": [], "employee_count": []}

with beam.Pipeline(options=PipelineOptions()) as pipeline:
    processed_data = (
        pipeline
        | "Read File" >> beam.io.ReadFromText('/content/sample - Sheet1.csv', skip_header_lines=1)
        | "Transform Data" >> beam.Map(lambda row: row.split(","))
        | "Filter Valid Rows" >> beam.Filter(lambda fields: len(fields) >= 6)
        | "Parse Data" >> beam.Map(lambda fields: (
            int(fields[0]),  # id
            fields[1],       # first_name
            fields[2],       # last_name
            f"{fields[1]} {fields[2]}",  # full_name
            int(fields[3]),  # age
            int(fields[4]),  # salary
            fields[5]        # department
        ))
    )

    # Total Salary Per Department
    total_salary_per_dept = (
        processed_data
        | "Map Salary to Dept" >> beam.Map(lambda x: (x[6], x[5]))
        | "Sum Salaries" >> beam.CombinePerKey(sum)
        | "Capture Total Salary" >> beam.ParDo(CaptureResults("total_salary"))
    )

    # Average Salary Per Department
    avg_salary_per_dept = (
        processed_data
        | "Map Dept to (Salary,1)" >> beam.Map(lambda x: (x[6], (x[5], 1)))
        | "Sum Salary & Count" >> beam.CombinePerKey(lambda vals: (sum(s for s, _ in vals), sum(c for _, c in vals)))
        | "Calculate Avg" >> beam.Map(lambda x: (x[0], x[1][0] / x[1][1]))
        | "Capture Avg Salary" >> beam.ParDo(CaptureResults("avg_salary"))
    )

    # Employee Count Per Department
    employee_count_per_dept = (
        processed_data
        | "Map Dept to 1" >> beam.Map(lambda x: (x[6], 1))
        | "Count Employees" >> beam.CombinePerKey(sum)
        | "Capture Employee Count" >> beam.ParDo(CaptureResults("employee_count"))
    )

    avg_salary_per_dept | "Print Results" >> beam.Map(print)




In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

# Function to store data from Beam pipeline
class CaptureResults(beam.DoFn):
    def process(self, element):
        yield element  # Emit element instead of using global dict

with beam.Pipeline(options=PipelineOptions()) as pipeline:
    processed_data = (
        pipeline
        | "Read File" >> beam.io.ReadFromText('/content/sample - Sheet1.csv', skip_header_lines=1)
        | "Transform Data" >> beam.Map(lambda row: row.split(","))
        | "Filter Valid Rows" >> beam.Filter(lambda fields: len(fields) >= 6)
        | "Parse Data" >> beam.Map(lambda fields: (
            int(fields[0]),  # id
            fields[1],       # first_name
            fields[2],       # last_name
            f"{fields[1]} {fields[2]}",  # full_name
            int(fields[3]),  # age
            int(fields[4]),  # salary
            fields[5]        # department
        ))
    )

    # Total Salary Per Department
    total_salary_per_dept = (
        processed_data
        | "Map Salary to Dept" >> beam.Map(lambda x: (x[6], x[5]))  # (department, salary)
        | "Sum Salaries" >> beam.CombinePerKey(sum)
        | "Capture Total Salary" >> beam.ParDo(CaptureResults())
    )

    # Average Salary Per Department
    avg_salary_per_dept = (
        processed_data
        | "Map Dept to (Salary,1)" >> beam.Map(lambda x: (x[6], (x[5], 1)))  # (dept, (salary, 1))
        | "Sum Salary & Count" >> beam.CombinePerKey(lambda vals: (sum(s for s, _ in vals), sum(c for _, c in vals)))
        | "Calculate Avg" >> beam.Map(lambda x: (x[0], x[1][0] / x[1][1]))  # (dept, avg_salary)
        | "Capture Avg Salary" >> beam.ParDo(CaptureResults())
    )

    # Employee Count Per Department
    employee_count_per_dept = (
        processed_data
        | "Map Dept to 1" >> beam.Map(lambda x: (x[6], 1))  # (dept, 1)
        | "Count Employees" >> beam.CombinePerKey(sum)
        | "Capture Employee Count" >> beam.ParDo(CaptureResults())
    )
    #salary per name
    salary_per_name = (
        processed_data
        | "Map Name to Salary" >> beam.Map(lambda x: (x[3], x[5]))  # (name, salary)
        | "Sum Salaries per name" >> beam.CombinePerKey(sum)
        | "Capture Salary Per Name" >> beam.ParDo(CaptureResults())
    )

    ## Collect and Print the Results
    #results = (
    #    {
    #        "total_salary": total_salary_per_dept,
    #        "avg_salary": avg_salary_per_dept,
    #        "employee_count": employee_count_per_dept,
    #    }
    #    #| "Combine All Results" >> beam.CoGroupByKey()
    #    #| "Print Results" >> beam.Map(print)
    #)

    salary_per_name | "Print Results" >> beam.Map(print)


('John Doe', 60000)
('Jane Smith', 55000)
('Michael Johnson', 75000)
('Emily Davis', 58000)
('Daniel Martinez', 90000)
('Sophia Lopez', 62000)
('James Gonzalez', 110000)
('Olivia Wilson', 70000)
('William Anderson', 63000)
('Ava Thomas', 59000)
('Alexander Taylor', 85000)
('Mia Moore', 54000)
('Ethan Jackson', 67000)
('Charlotte White', 72000)
('Benjamin Harris', 78000)
('Amelia Clark', 64000)
('Lucas Robinson', 98000)
('Harper Walker', 66000)
('Mason Hall', 74000)
('Evelyn Allen', 60000)


In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
import random
import heapq

# Function to store data from Beam pipeline
class CaptureResults(beam.DoFn):
    def process(self, element):
        yield element  # Emit element instead of using a global dict

# Custom Combiner for Sampling
class SampleSalaries(beam.CombineFn):
    def __init__(self, sample_size=3):  # Change `sample_size` as needed
        self.sample_size = sample_size

    def create_accumulator(self):
        return []

    def add_input(self, accumulator, element):
        accumulator.append(element)
        return accumulator

    def merge_accumulators(self, accumulators):
        combined = sum(accumulators, [])
        return combined

    def extract_output(self, accumulator):
        return random.sample(accumulator, min(len(accumulator), self.sample_size))

# Custom Combiner for Top N Salaries
class TopNSalaries(beam.CombineFn):
    def __init__(self, top_n=3):  # Change `top_n` as needed
        self.top_n = top_n

    def create_accumulator(self):
        return []

    def add_input(self, accumulator, element):
        heapq.heappush(accumulator, element)
        if len(accumulator) > self.top_n:
            heapq.heappop(accumulator)
        return accumulator

    def merge_accumulators(self, accumulators):
        combined = []
        for acc in accumulators:
            for item in acc:
                heapq.heappush(combined, item)
                if len(combined) > self.top_n:
                    heapq.heappop(combined)
        return combined

    def extract_output(self, accumulator):
        return sorted(accumulator, reverse=True)  # Return top N salaries sorted in descending order

with beam.Pipeline(options=PipelineOptions()) as pipeline:
    processed_data = (
        pipeline
        | "Read File" >> beam.io.ReadFromText('/content/sample - Sheet1.csv', skip_header_lines=1)
        | "Transform Data" >> beam.Map(lambda row: row.split(","))
        | "Filter Valid Rows" >> beam.Filter(lambda fields: len(fields) >= 6)
        | "Parse Data" >> beam.Map(lambda fields: (
            int(fields[0]),  # id
            fields[1],       # first_name
            fields[2],       # last_name
            f"{fields[1]} {fields[2]}",  # full_name
            int(fields[3]),  # age
            int(fields[4]),  # salary
            fields[5]        # department
        ))
    )

    # Total Salary Per Department
    total_salary_per_dept = (
        processed_data
        | "Map Salary to Dept" >> beam.Map(lambda x: (x[6], x[5]))  # (department, salary)
        | "Sum Salaries" >> beam.CombinePerKey(sum)
        | "Capture Total Salary" >> beam.ParDo(CaptureResults())
    )

    # Max Salary Per Department
    max_salary_per_dept = (
        processed_data
        | "Map Salary Max" >> beam.Map(lambda x: (x[6], x[5]))
        | "Max Salary" >> beam.CombinePerKey(max)
        | "Capture Max Salary" >> beam.ParDo(CaptureResults())
    )

    # Min Salary Per Department
    min_salary_per_dept = (
        processed_data
        | "Map Salary Min" >> beam.Map(lambda x: (x[6], x[5]))
        | "Min Salary" >> beam.CombinePerKey(min)
        | "Capture Min Salary" >> beam.ParDo(CaptureResults())
    )

    # Average Salary Per Department
    avg_salary_per_dept = (
        processed_data
        | "Map Dept to (Salary,1)" >> beam.Map(lambda x: (x[6], (x[5], 1)))  # (dept, (salary, 1))
        | "Sum Salary & Count" >> beam.CombinePerKey(lambda vals: (sum(s for s, _ in vals), sum(c for _, c in vals)))
        | "Calculate Avg" >> beam.Map(lambda x: (x[0], x[1][0] / x[1][1]))  # (dept, avg_salary)
        | "Capture Avg Salary" >> beam.ParDo(CaptureResults())
    )

    # Latest Salary Per Department (assuming highest salary is the latest)
    latest_salary_per_dept = (
        processed_data
        | "Map Latest Salary" >> beam.Map(lambda x: (x[6], x[5]))  # (dept, salary)
        | "Latest Salary" >> beam.CombinePerKey(lambda vals: max(vals))  # Take max value as latest
        | "Capture Latest Salary" >> beam.ParDo(CaptureResults())
    )

    # Sample Salaries Per Department
    sample_salaries_per_dept = (
        processed_data
        | "Map Sample Salaries" >> beam.Map(lambda x: (x[6], x[5]))
        | "Sample Salaries" >> beam.CombinePerKey(SampleSalaries(sample_size=3))
        | "Capture Sample Salaries" >> beam.ParDo(CaptureResults())
    )

    # Top 3 Salaries Per Department
    top_salaries_per_dept = (
        processed_data
        | "Map Top Salaries" >> beam.Map(lambda x: (x[6], x[5]))
        | "Top 3 Salaries" >> beam.CombinePerKey(TopNSalaries(top_n=4))
        | "Capture Top Salaries" >> beam.ParDo(CaptureResults())
    )

    #Top 3 Salaries per Name
    top_salaries_per_name = (
        processed_data
        | "Map Top Salaries per Name" >> beam.Map(lambda x : (x[3], x[5]))
        | "Top 3 Salaries per Name " >> beam.CombinePerKey(TopNSalaries(top_n =1))
        | "Capture Top Salaries per Name" >> beam.ParDo(CaptureResults())
    )

    ## Collect and Print the Results
    #results = (
    #    {
    #        "total_salary": total_salary_per_dept,
    #        "max_salary": max_salary_per_dept,
    #        "min_salary": min_salary_per_dept,
    #        "avg_salary": avg_salary_per_dept,
    #        "latest_salary": latest_salary_per_dept,
    #        "sample_salaries": sample_salaries_per_dept,
    #        "top_salaries": top_salaries_per_dept,
    #    }
    #    | "Combine All Results" >> beam.CoGroupByKey()
    #    | "Print Results" >> beam.Map(print)
    #)
    top_salaries_per_dept | "Print Results" >> beam.Map(print)


('IT', [98000, 90000, 67000, 63000])
('HR', [72000, 66000, 62000, 59000])
('Finance', [110000, 85000, 78000, 75000])
('Marketing', [70000, 64000, 60000, 58000])


Top employee from each department who have maximum salary

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

# Custom DoFn to filter top employees
class FilterTopEmployees(beam.DoFn):
    def process(self, element):
        dept, grouped_data = element  # grouped_data = {'employees': [(salary, employee)], 'max_salary': [highest_salary]}

        employees = grouped_data.get("employees", [])
        max_salary_list = grouped_data.get("max_salary", [])

        if max_salary_list:  # Ensure max_salary_list is not empty
            max_salary = max_salary_list[0]  # Extract the max salary
            for salary, employee in employees:
                if salary == max_salary:  # Keep only employees with max salary
                    yield employee

# Define Apache Beam Pipeline
with beam.Pipeline(options=PipelineOptions()) as pipeline:
    processed_data = (
        pipeline
        | "Read File" >> beam.io.ReadFromText('/content/sample - Sheet1.csv', skip_header_lines=1)
        | "Transform Data" >> beam.Map(lambda row: row.split(","))
        | "Filter Valid Rows" >> beam.Filter(lambda fields: len(fields) >= 6)
        | "Parse Data" >> beam.Map(lambda fields: (
            int(fields[0]),  # id
            fields[1],       # first_name
            fields[2],       # last_name
            f"{fields[1]} {fields[2]}",  # full_name
            int(fields[3]),  # age
            int(fields[4]),  # salary
            fields[5]        # department
        ))
    )

    # Step 1: Compute the highest salary per department
    max_salary_per_dept = (
        processed_data
        | "Extract (Dept, Salary)" >> beam.Map(lambda x: (x[6], x[5]))  # (department, salary)
        | "Find Max Salary Per Dept" >> beam.CombinePerKey(max)  # Get max salary per department
    )

    # Step 2: Map employees to their (Dept, (Salary, Employee)) tuple
    employees_grouped_by_dept = (
        processed_data
        | "Map Employee to (Dept, (Salary, Employee))" >> beam.Map(lambda x: (x[6], (x[5], x)))  # (department, (salary, employee))
    )

    # Step 3: Group employees and max salary together
    grouped_data = (
        {
            "employees": employees_grouped_by_dept,
            "max_salary": max_salary_per_dept
        }
        | "Group Employees & Max Salary" >> beam.CoGroupByKey()  # (dept, {'employees': [(salary, employee)], 'max_salary': [highest_salary]})
    )

    # Step 4: Filter top employees per department
    top_employee_per_dept = (
        grouped_data
        | "Filter Top Employees" >> beam.ParDo(FilterTopEmployees())  # Keep only employees with max salary
    )

    # Print the top employee per department
    grouped_data | "Print Top Employee" >> beam.Map(print)


(17, 'Lucas', 'Robinson', 'Lucas Robinson', 42, 98000, 'IT')
(14, 'Charlotte', 'White', 'Charlotte White', 33, 72000, 'HR')
(7, 'James', 'Gonzalez', 'James Gonzalez', 45, 110000, 'Finance')
(8, 'Olivia', 'Wilson', 'Olivia Wilson', 32, 70000, 'Marketing')


top 3 highest paying employee from each depatment department

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

# Custom DoFn to filter top 3 employees per department
class FilterTop3Employees(beam.DoFn):
    def process(self, element):
        dept, employees = element  # (department, [(salary, employee)])

        # Sort employees by salary in descending order and take top 3
        top_3_employees = sorted(employees, key=lambda x: x[0], reverse=True)[:3]

        for _, employee in top_3_employees:
            yield employee

# Define Apache Beam Pipeline
with beam.Pipeline(options=PipelineOptions()) as pipeline:
    processed_data = (
        pipeline
        | "Read File" >> beam.io.ReadFromText('/content/sample - Sheet1.csv', skip_header_lines=1)
        | "Transform Data" >> beam.Map(lambda row: row.split(","))
        | "Filter Valid Rows" >> beam.Filter(lambda fields: len(fields) >= 6)
        | "Parse Data" >> beam.Map(lambda fields: (
            int(fields[0]),  # id
            fields[1],       # first_name
            fields[2],       # last_name
            f"{fields[1]} {fields[2]}",  # full_name
            int(fields[3]),  # age
            int(fields[4]),  # salary
            fields[5]        # department
        ))
    )

    # Step 1: Group employees by department with salaries
    employees_grouped_by_dept = (
        processed_data
        | "Map Employee to (Dept, (Salary, Employee))" >> beam.Map(lambda x: (x[6], (x[5], x)))  # (department, (salary, employee))
        | "Group By Department" >> beam.GroupByKey()  # (dept, [(salary, employee), (salary, employee), ...])
    )

    # Step 2: Extract the top 3 employees per department
    top_3_employees_per_dept = (
        employees_grouped_by_dept
        | "Filter Top 3 Employees" >> beam.ParDo(FilterTop3Employees())  # Keep only top 3 employees per dept
    )

    # Print the top 3 employees per department
    top_3_employees_per_dept | "Print Top 3 Employees" >> beam.Map(print)


(17, 'Lucas', 'Robinson', 'Lucas Robinson', 42, 98000, 'IT')
(5, 'Daniel', 'Martinez', 'Daniel Martinez', 40, 90000, 'IT')
(13, 'Ethan', 'Jackson', 'Ethan Jackson', 31, 67000, 'IT')
(14, 'Charlotte', 'White', 'Charlotte White', 33, 72000, 'HR')
(18, 'Harper', 'Walker', 'Harper Walker', 30, 66000, 'HR')
(6, 'Sophia', 'Lopez', 'Sophia Lopez', 27, 62000, 'HR')
(7, 'James', 'Gonzalez', 'James Gonzalez', 45, 110000, 'Finance')
(11, 'Alexander', 'Taylor', 'Alexander Taylor', 38, 85000, 'Finance')
(15, 'Benjamin', 'Harris', 'Benjamin Harris', 37, 78000, 'Finance')
(8, 'Olivia', 'Wilson', 'Olivia Wilson', 32, 70000, 'Marketing')
(16, 'Amelia', 'Clark', 'Amelia Clark', 29, 64000, 'Marketing')
(20, 'Evelyn', 'Allen', 'Evelyn Allen', 27, 60000, 'Marketing')


top 3 lowest paying employee from each depatment

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

# Custom DoFn to filter top 3 employees per department
class FilterTop3Employees(beam.DoFn):
    def process(self, element):
        dept, employees = element  # (department, [(salary, employee)])

        # Sort employees by salary in ascending order and take top 3
        bottom_3_employees = sorted(employees, key=lambda x: x[0])[:3]

        for _, employee in bottom_3_employees:
            yield employee

# Define Apache Beam Pipeline
with beam.Pipeline(options=PipelineOptions()) as pipeline:
    processed_data = (
        pipeline
        | "Read File" >> beam.io.ReadFromText('/content/sample - Sheet1.csv', skip_header_lines=1)
        | "Transform Data" >> beam.Map(lambda row: row.split(","))
        | "Filter Valid Rows" >> beam.Filter(lambda fields: len(fields) >= 6)
        | "Parse Data" >> beam.Map(lambda fields: (
            int(fields[0]),  # id
            fields[1],       # first_name
            fields[2],       # last_name
            f"{fields[1]} {fields[2]}",  # full_name
            int(fields[3]),  # age
            int(fields[4]),  # salary
            fields[5]        # department
        ))
    )

    # Step 1: Group employees by department with salaries
    employees_grouped_by_dept = (
        processed_data
        | "Map Employee to (Dept, (Salary, Employee))" >> beam.Map(lambda x: (x[6], (x[5], x)))  # (department, (salary, employee))
        | "Group By Department" >> beam.GroupByKey()  # (dept, [(salary, employee), (salary, employee), ...])
    )

    # Step 2: Extract the top 3 employees per department
    top_3_employees_per_dept = (
        employees_grouped_by_dept
        | "Filter Top 3 Employees" >> beam.ParDo(FilterTop3Employees())  # Keep only top 3 employees per dept
    )

    # Print the top 3 employees per department
    top_3_employees_per_dept | "Print Top 3 Employees" >> beam.Map(print)


(1, 'John', 'Doe', 'John Doe', 30, 60000, 'IT')
(9, 'William', 'Anderson', 'William Anderson', 29, 63000, 'IT')
(13, 'Ethan', 'Jackson', 'Ethan Jackson', 31, 67000, 'IT')
(2, 'Jane', 'Smith', 'Jane Smith', 25, 55000, 'HR')
(10, 'Ava', 'Thomas', 'Ava Thomas', 26, 59000, 'HR')
(6, 'Sophia', 'Lopez', 'Sophia Lopez', 27, 62000, 'HR')
(19, 'Mason', 'Hall', 'Mason Hall', 34, 74000, 'Finance')
(3, 'Michael', 'Johnson', 'Michael Johnson', 35, 75000, 'Finance')
(15, 'Benjamin', 'Harris', 'Benjamin Harris', 37, 78000, 'Finance')
(12, 'Mia', 'Moore', 'Mia Moore', 24, 54000, 'Marketing')
(4, 'Emily', 'Davis', 'Emily Davis', 28, 58000, 'Marketing')
(20, 'Evelyn', 'Allen', 'Evelyn Allen', 27, 60000, 'Marketing')
